# This notebook test full workflow while using MLRun packgers

This notebook will test :

1. Loging object to MLRun such as: `pd.DataFrame, np.ndarray, str and list`.
2. Unpack `pd.DataFrame, np.ndarray, str and list` and compare them to the original values.
3. Logging object to MLRun such as: `pd.DataFrame, np.ndarray, string, list` with non default artifact type and file format.
4. Unpack outputs from step 3 and validate the expcted values are matched to the original 
5. Pack Custom packager that pack a zip file path to a zip file
6. Unpack Custom packager

In [ ]:
import mlrun
import pandas as pd
import numpy as np
import os
import shutil
from mlrun.package.log_hint import LogHint

In [3]:
project = mlrun.get_or_create_project("test-pack",'./context',user_project=True)

> 2026-03-08 13:49:25,891 [info] Created and saved project: {"context":"./context","from_template":null,"name":"test-pack-shapira","overwrite":false,"save":true}
> 2026-03-08 13:49:25,892 [info] Project created successfully: {"project_name":"test-pack-shapira","stored_in_db":true}


In [5]:
project.spec.custom_packagers

[]

## 1.  Loging object to MLRun such as: `pd.DataFrame, np.array, string, list`

Expcted logging result:
* `pd.DataFrame` - as dataset artifact type and default parquet file
* `np.nparray` - as artifact artifact type and default npy file
* `str` - as a run result 
* `list` as a run result

In [6]:
%%writefile ./context/generate-outputs.py
import numpy as np
import pandas as pd


def generate_outputs():
    df_true = pd.DataFrame(
        data={
            **{f"column_{i}": np.arange(0, 1000) for i in range(1, 10)},
        },
    ).set_index(keys=["column_7", "column_8", "column_9"])

    np_example = np.array(
        [[[1, 2, 3], [4, 5, 6], [7, 8, 9]], [[1, 2, 3], [4, 5, 6], [7, 8, 9]]]
    )

    example_string = "Example_String"

    example_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, "A"]
    return df_true, np_example,  example_string ,example_list, df_true


Writing ./context/generate-outputs.py


In [7]:
project.set_function("./generate-outputs.py",name='generate-outputs',kind='job',image='mlrun/mlrun',handler='generate_outputs')

In [8]:
test_artifact_path = mlrun.mlconf.artifact_path + "/test"

In [10]:
df_loghint = LogHint(key="my-df",tag="v2",artifact_path=test_artifact_path,packing_kwargs={"file_format":"csv"},labels={"t":1})

In [11]:
outputs_run = project.run_function("generate-outputs",returns=["df","np_example","example_string","example_list",df_loghint])

> 2026-03-08 13:49:25,989 [info] Storing function: {"db":"http://mlrun-api:8080","name":"generate-outputs-generate-outputs","uid":"10992263491349d583b335b337b8713c"}


Unexpected run keyword argument 'local' was ignored.


> 2026-03-08 13:49:26,258 [info] Job is running in the background, pod: generate-outputs-generate-outputs-dh8c4
> 2026-03-08 13:49:29,556 [info] To track results use the CLI: {"info_cmd":"mlrun get run 10992263491349d583b335b337b8713c -p test-pack-shapira","logs_cmd":"mlrun logs 10992263491349d583b335b337b8713c -p test-pack-shapira"}
> 2026-03-08 13:49:29,556 [info] Or click for UI: {"ui_url":"https://dashboard.default-tenant.app.cust-cs-il.iguazio-cd0.com/mlprojects/test-pack-shapira/jobs/monitor-jobs/generate-outputs-generate-outputs/10992263491349d583b335b337b8713c/overview"}
> 2026-03-08 13:49:29,557 [info] Run execution finished: {"name":"generate-outputs-generate-outputs","status":"completed"}


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifacts
test-pack-shapira,...37b8713c,0,Mar 08 13:49:29,2026-03-08 13:49:29.545118+00:00,completed,run,generate-outputs-generate-outputs,v3io_user=shapirakind=jobowner=shapiramlrun/client_version=1.11.0-rc40mlrun/client_python_version=3.11.14host=generate-outputs-generate-outputs-dh8c4,,,example_string=Example_Stringexample_list[0]=1[1]=2[2]=3[3]=4[4]=5[5]=6[6]=7[7]=8[8]=9[9]=A,my-dfmy-dfnp_exampledf


> 2026-03-08 13:49:39,404 [info] Run execution finished: {"name":"generate-outputs-generate-outputs","status":"completed"}


**Validate Run Result**

In [12]:
df_outputs_run = project.get_store_resource(outputs_run.outputs["my-df"])
assert df_outputs_run.kind == 'dataset'
assert df_outputs_run.target_path.endswith("csv")
assert df_outputs_run.target_path == 'v3io:///projects/test-pack-shapira/artifacts/test/generate-outputs-generate-outputs/0/my-df.csv'
assert df_outputs_run.labels == {'t': '1'}

In [13]:
df_outputs_run = project.get_store_resource(outputs_run.outputs["df"])
assert df_outputs_run.kind == 'dataset'
assert df_outputs_run.target_path.endswith("parquet")

In [14]:
df_outputs_run = project.get_store_resource(outputs_run.outputs["np_example"])
assert df_outputs_run.kind == 'artifact'
assert df_outputs_run.target_path.endswith("npy")

In [15]:
assert outputs_run.outputs["example_string"]=='Example_String'
assert outputs_run.outputs["example_list"]==[1, 2, 3, 4, 5, 6, 7, 8, 9, 'A']

## 2. Unpack `pd.DataFrame, np.ndarray, str and list` and compare them to the original values.

Expected result - 
* run the function without any assertion errors and validate the numpy and pandas are unpacked as expcted 

In [17]:
%%writefile ./context/generate-inputs.py
import numpy as np
import pandas as pd


def generate_inputs(np_example: np.ndarray, df: pd.DataFrame):
    df_true = pd.DataFrame(
        data={
            **{f"column_{i}": np.arange(0, 1000) for i in range(1, 10)},
        },
    ).set_index(keys=["column_7", "column_8", "column_9"])

    # compare dataframe unpack
    assert df_true.compare(df).shape == (0, 0)

    # compare dataframe unpack
    np_true = np.array(
        [[[1, 2, 3], [4, 5, 6], [7, 8, 9]], [[1, 2, 3], [4, 5, 6], [7, 8, 9]]]
    )
    assert np.array_equal(np_true, np_example)

    return "OK"

Writing ./context/generate-inputs.py


In [18]:
project.set_function("./generate-inputs.py",name='generate-inputs',kind='job',image='mlrun/mlrun',handler='generate_inputs')

In [19]:
df = outputs_run.outputs["df"]
array_np = outputs_run.outputs["np_example"]


In [20]:
state_loghint = LogHint(key="state")

In [21]:
inputs_run = project.run_function("generate-inputs",inputs={"df":df,"np_example":array_np},returns=[state_loghint])

> 2026-03-08 13:49:39,630 [info] Storing function: {"db":"http://mlrun-api:8080","name":"generate-inputs-generate-inputs","uid":"16a079e2af544017b2d60228d9b6e3c1"}


Unexpected run keyword argument 'local' was ignored.


> 2026-03-08 13:49:39,901 [info] Job is running in the background, pod: generate-inputs-generate-inputs-bvpzl
> 2026-03-08 13:49:42,920 [info] downloading v3io:///projects/test-pack-shapira/artifacts/generate-outputs-generate-outputs/0/np_example.npy to local temp file
> 2026-03-08 13:49:43,047 [info] To track results use the CLI: {"info_cmd":"mlrun get run 16a079e2af544017b2d60228d9b6e3c1 -p test-pack-shapira","logs_cmd":"mlrun logs 16a079e2af544017b2d60228d9b6e3c1 -p test-pack-shapira"}
> 2026-03-08 13:49:43,047 [info] Or click for UI: {"ui_url":"https://dashboard.default-tenant.app.cust-cs-il.iguazio-cd0.com/mlprojects/test-pack-shapira/jobs/monitor-jobs/generate-inputs-generate-inputs/16a079e2af544017b2d60228d9b6e3c1/overview"}
> 2026-03-08 13:49:43,048 [info] Run execution finished: {"name":"generate-inputs-generate-inputs","status":"completed"}


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results
test-pack-shapira,...d9b6e3c1,0,Mar 08 13:49:42,2026-03-08 13:49:43.036029+00:00,completed,run,generate-inputs-generate-inputs,v3io_user=shapirakind=jobowner=shapiramlrun/client_version=1.11.0-rc40mlrun/client_python_version=3.11.14host=generate-inputs-generate-inputs-bvpzl,dfnp_example,,state=OK


> 2026-03-08 13:49:51,019 [info] Run execution finished: {"name":"generate-inputs-generate-inputs","status":"completed"}


**Validate Run Result**

In [22]:
assert inputs_run.outputs["state"] == "OK"

## 3. Logging object to MLRun such as: `pd.DataFrame, np.ndarray, string, list` with non default artifact type and file format.

Excpected output result - 

* df_pkl as an artifact artifact type and pickle file
* df_csv as an artifact artifact type and a csv file
* np_csv as an artifact artifact type and a csv file

In [24]:
%%writefile ./context/generate-outputs-non-default.py
import numpy as np
import pandas as pd
from numpy import asarray


def generate_outputs():
    df_true = pd.DataFrame(
        data={
            **{f"column_{i}": np.arange(0, 1000) for i in range(1, 10)},
        },
    ).set_index(keys=["column_7", "column_8", "column_9"])

    np_example = asarray([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]])

    df_pkl = df_true
    df_csv = df_true
    np_csv = np_example
    return df_pkl, df_csv, np_csv

Writing ./context/generate-outputs-non-default.py


In [25]:
project.set_function("./generate-outputs-non-default.py",name='generate-outputs-non-default',kind='job',image='mlrun/mlrun',handler='generate_outputs')

In [26]:
pkl_loghint = LogHint(key="df_pkl",artifact_type=mlrun.ArtifactType.OBJECT)
df_csv_loghint = LogHint(key="df_csv",artifact_type=mlrun.ArtifactType.FILE,packing_kwargs={"file_format":"csv"})
np_csv_loghint = LogHint(key="np_csv",artifact_type=mlrun.ArtifactType.DATASET,packing_kwargs={"file_format":"csv"})

In [27]:
non_defualt_run = project.run_function('generate-outputs-non-default',returns=[pkl_loghint,df_csv_loghint,np_csv_loghint])

> 2026-03-08 13:49:51,102 [info] Storing function: {"db":"http://mlrun-api:8080","name":"generate-outputs-non-default-generate-outputs","uid":"abcca6622337421d8c2dc2e4849b3da2"}


Unexpected run keyword argument 'local' was ignored.


> 2026-03-08 13:49:51,368 [info] Job is running in the background, pod: generate-outputs-non-default-generate-outputs-jzp6p
> 2026-03-08 13:49:54,670 [info] To track results use the CLI: {"info_cmd":"mlrun get run abcca6622337421d8c2dc2e4849b3da2 -p test-pack-shapira","logs_cmd":"mlrun logs abcca6622337421d8c2dc2e4849b3da2 -p test-pack-shapira"}
> 2026-03-08 13:49:54,670 [info] Or click for UI: {"ui_url":"https://dashboard.default-tenant.app.cust-cs-il.iguazio-cd0.com/mlprojects/test-pack-shapira/jobs/monitor-jobs/generate-outputs-non-default-generate-outputs/abcca6622337421d8c2dc2e4849b3da2/overview"}
> 2026-03-08 13:49:54,670 [info] Run execution finished: {"name":"generate-outputs-non-default-generate-outputs","status":"completed"}


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifacts
test-pack-shapira,...849b3da2,0,Mar 08 13:49:54,2026-03-08 13:49:54.658664+00:00,completed,run,generate-outputs-non-default-generate-outputs,v3io_user=shapirakind=jobowner=shapiramlrun/client_version=1.11.0-rc40mlrun/client_python_version=3.11.14host=generate-outputs-non-default-generate-outputs-jzp6p,,,,np_csvdf_csvdf_pkl


> 2026-03-08 13:50:02,497 [info] Run execution finished: {"name":"generate-outputs-non-default-generate-outputs","status":"completed"}


**Validate Run outputs**

In [28]:
df_pkl_non_defualt_run = project.get_store_resource(non_defualt_run.outputs["df_pkl"])
assert df_pkl_non_defualt_run.kind == 'artifact'
assert df_pkl_non_defualt_run.target_path.endswith("pkl")

In [29]:
df_csv_non_defualt_run = project.get_store_resource(non_defualt_run.outputs["df_csv"])
assert df_csv_non_defualt_run.kind == 'artifact'
assert df_csv_non_defualt_run.target_path.endswith("csv")

In [30]:
np_csv_non_defualt_run = project.get_store_resource(non_defualt_run.outputs["np_csv"])
assert np_csv_non_defualt_run.kind == 'dataset'
assert np_csv_non_defualt_run.target_path.endswith("csv")

## 4. Unpack outputs from step 3 and validate the expcted values are matched to the original 

Excpted result -
* run without assertion errors.

In [31]:
%%writefile ./context/generate-inputs-non-defaults.py
import numpy as np
import pandas as pd
from numpy import asarray

def generate_inputs(df_pkl: pd.DataFrame, df_csv: pd.DataFrame, np_csv: np.ndarray):
    df_true = pd.DataFrame(
        data={
            **{f"column_{i}": np.arange(0, 1000) for i in range(1, 10)},
        },
    ).set_index(keys=["column_7", "column_8", "column_9"])
    # compare dataframe unpack
    assert df_true.compare(df_pkl).shape == (0, 0)
    assert df_true.compare(df_csv).shape == (0, 0)
    # compare dataframe unpack
    np_true = asarray([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

    assert np.array_equal(np_true, np_csv)

    return "OK"


Writing ./context/generate-inputs-non-defaults.py


In [32]:
df_pkl = non_defualt_run.outputs["df_pkl"]
df_csv = non_defualt_run.outputs["df_csv"]
np_csv = non_defualt_run.outputs["np_csv"]

In [33]:
project.set_function("./generate-inputs-non-defaults.py",name='generate-inputs-non-defaults',kind='job',image='mlrun/mlrun',handler='generate_inputs')

In [34]:
inputs_run_non_defualt = project.run_function("generate-inputs-non-defaults",inputs={"df_pkl":df_pkl,"df_csv":df_csv,"np_csv":np_csv},returns=[state_loghint])

> 2026-03-08 13:50:02,707 [info] Storing function: {"db":"http://mlrun-api:8080","name":"generate-inputs-non-defaults-generate-inputs","uid":"28336156a1814e7bab0e12a8c2115f31"}


Unexpected run keyword argument 'local' was ignored.


> 2026-03-08 13:50:02,977 [info] Job is running in the background, pod: generate-inputs-non-defaults-generate-inputs-fbngs
> 2026-03-08 13:50:06,251 [info] downloading v3io:///projects/test-pack-shapira/artifacts/generate-outputs-non-default-generate-outputs/0/df_pkl.pkl to local temp file
> 2026-03-08 13:50:06,275 [info] downloading v3io:///projects/test-pack-shapira/artifacts/generate-outputs-non-default-generate-outputs/0/df_csv.csv to local temp file
> 2026-03-08 13:50:06,401 [info] To track results use the CLI: {"info_cmd":"mlrun get run 28336156a1814e7bab0e12a8c2115f31 -p test-pack-shapira","logs_cmd":"mlrun logs 28336156a1814e7bab0e12a8c2115f31 -p test-pack-shapira"}
> 2026-03-08 13:50:06,401 [info] Or click for UI: {"ui_url":"https://dashboard.default-tenant.app.cust-cs-il.iguazio-cd0.com/mlprojects/test-pack-shapira/jobs/monitor-jobs/generate-inputs-non-defaults-generate-inputs/28336156a1814e7bab0e12a8c2115f31/overview"}
> 2026-03-08 13:50:06,402 [info] Run execution finished:

project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results
test-pack-shapira,...c2115f31,0,Mar 08 13:50:05,2026-03-08 13:50:06.390188+00:00,completed,run,generate-inputs-non-defaults-generate-inputs,v3io_user=shapirakind=jobowner=shapiramlrun/client_version=1.11.0-rc40mlrun/client_python_version=3.11.14host=generate-inputs-non-defaults-generate-inputs-fbngs,df_pkldf_csvnp_csv,,state=OK


> 2026-03-08 13:50:12,097 [info] Run execution finished: {"name":"generate-inputs-non-defaults-generate-inputs","status":"completed"}


## 6. Pack Custom packager that pack a zip file path to a zip file

Expcted result -
* to pack a string as an artifact type artifact and as a zip file

In [36]:
os.makedirs('./custom_pack/',exist_ok=True)

Writing custome object file

In [38]:
%%writefile ./custom_pack/custom.py
import os
import tempfile
import pathlib
from typing import Tuple

from mlrun import ArtifactType
from mlrun.artifacts import Artifact
from mlrun.datastore import DataItem
from mlrun.errors import MLRunInvalidArgumentError
from mlrun.package import DefaultPackager
from mlrun.package.utils import DEFAULT_ARCHIVE_FORMAT, ArchiveSupportedFormat


class StringPackager(DefaultPackager):
    """
    ``builtins.str`` packager.
    """

    PACKABLE_OBJECT_TYPE = str
    DEFAULT_PACKING_ARTIFACT_TYPE = ArtifactType.FILE
    DEFAULT_UNPACKING_ARTIFACT_TYPE = ArtifactType.PATH

    def pack_file(
        self, obj: str, key: str, archive_format: str = DEFAULT_ARCHIVE_FORMAT
    ) -> Tuple[Artifact, dict]:
        """
        Pack a zip value content (pack the file or directory in that path).

        :param obj:            The zip object to pack.
        :param key:            The key to use for the artifact.
        :param archive_format: The archive format to use in case the path is of a directory. Default is zip.

        :return: The packed artifact and instructions.
        """
        # Proceed by path type (file or directory):
        if os.path.isfile(obj):
            # Create the artifact:
            artifact = Artifact(key=key, src_path=os.path.abspath(obj))
            instructions = {"archive_format": "zip", "is_directory": True}
        else:
            raise MLRunInvalidArgumentError(
                f"The given path is not a file nor a directory: '{obj}'"
            )

        return artifact, instructions

    def unpack_file(
        self, data_item: DataItem, is_directory: bool = False, archive_format: str = None
    ) -> str:
        """
        Unpack a data item representing a path string. If the path is of a file, the file is downloaded to a local
        temporary directory and its path is returned. If the path is of a directory, the archive is extracted and the
        directory path extracted is returned.

        :param data_item:      The data item to unpack.
        :param is_directory:   Whether the path should be treated as a file or a directory. Files (even archives like
                               zip) won't be extracted.
        :param archive_format: The archive format to use in case the path is of a directory. Default is None - will be
                               read by the archive file extension.

        :return: The unpacked string.
        """
        # Get the file to a local temporary directory:
        path = data_item.local()

        # Mark the downloaded file for future clear:
        self.add_future_clearing_path(path=path)

        # If it's not a directory, return the file path. Otherwise, it should be extracted according to the archive
        # format:
        if not is_directory:
            return path

        # Get the archive format by the file extension:
        if archive_format is None:
            archive_format = ArchiveSupportedFormat.match_format(path=path)
        if archive_format is None:
            raise MLRunInvalidArgumentError(
                f"Archive format of {data_item.key} ('{''.join(pathlib.Path(path).suffixes)}') is not supported. "
                f"Supported formats are: {' '.join(ArchiveSupportedFormat.get_all_formats())}"
            )

        # Extract the archive:
        archiver = ArchiveSupportedFormat.get_format_handler(fmt=archive_format)
        directory_path = archiver.extract_archive(
            archive_path=path, output_path=os.path.dirname(path)
        )

        # Mark the extracted content for future clear:
        self.add_future_clearing_path(path=directory_path)

        # Return the extracted directory path:
        return directory_path

Writing ./custom_pack/custom.py


In [39]:
%%writefile ./custom_pack/test-custom.py
import os
import shutil
import time

from custom import StringPackager


def func_custom():
    os.makedirs("./files/")
    with open("./files/test_1.txt", "w") as file:
        file.write("file1")
        file.close()

    with open("./files/test_2.txt", "w") as file:
        file.write("file2")
        file.close()

    with open("./files/test_3.txt", "w") as file:
        file.write("file3")
        file.close()

    shutil.make_archive("test", "zip", "./files/")
    return "./test.zip"

Writing ./custom_pack/test-custom.py


In [40]:
%%writefile ./custom_pack/test-custom-unpack.py
import os
import shutil
import time

from custom import StringPackager

from mlrun import ArtifactType


def func_custom(zip_file: str):
    print(zip_file)
    with open(f"{zip_file}/test_1.txt", "r") as file:
        read = file.read()
        assert read == "file1"
        file.close()

    with open(f"{zip_file}/test_2.txt", "r") as file:
        read = file.read()
        assert read == "file2"
        file.close()

    with open(f"{zip_file}/test_3.txt", "r") as file:
        read = file.read()
        assert read == "file3"
        file.close()

    return "Completed"

Writing ./custom_pack/test-custom-unpack.py


In [42]:
project.add_custom_packager(packager="custom.StringPackager",is_mandatory=True)

In [43]:
shutil.make_archive('test','zip','./custom_pack/')

'/User/General_Testing/packagers/test.zip'

In [44]:
source_art = project.log_artifact("source-zip",local_path='./test.zip')

In [45]:
project.set_source(source_art.target_path,pull_at_runtime=True)

In [46]:
project.save()

In [47]:
func = project.set_function("./test-custom.py",name="test-custom",kind='job',image='mlrun/mlrun',handler='func_custom',with_repo=True)

In [48]:
custom_loghint = LogHint(key="test_file",artifact_type=mlrun.ArtifactType.FILE,labels={"custom":"true"})

In [49]:
test_func = func.run(returns=[custom_loghint])

> 2026-03-08 13:50:13,307 [info] Storing function: {"db":"http://mlrun-api:8080","name":"test-custom-func-custom","uid":"887278fabddd4115aae969354a7a6fbe"}


Unexpected run keyword argument 'local' was ignored.


> 2026-03-08 13:50:13,568 [info] Job is running in the background, pod: test-custom-func-custom-gkvz2
> 2026-03-08 13:50:16,675 [info] extracting source from v3io:///projects/test-pack-shapira/artifacts/source-zip.zip to /mlrun/code
> 2026-03-08 13:50:16,979 [info] To track results use the CLI: {"info_cmd":"mlrun get run 887278fabddd4115aae969354a7a6fbe -p test-pack-shapira","logs_cmd":"mlrun logs 887278fabddd4115aae969354a7a6fbe -p test-pack-shapira"}
> 2026-03-08 13:50:16,979 [info] Or click for UI: {"ui_url":"https://dashboard.default-tenant.app.cust-cs-il.iguazio-cd0.com/mlprojects/test-pack-shapira/jobs/monitor-jobs/test-custom-func-custom/887278fabddd4115aae969354a7a6fbe/overview"}
> 2026-03-08 13:50:16,979 [info] Run execution finished: {"name":"test-custom-func-custom","status":"completed"}


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifacts
test-pack-shapira,...4a7a6fbe,0,Mar 08 13:50:16,2026-03-08 13:50:16.966829+00:00,completed,run,test-custom-func-custom,v3io_user=shapirakind=jobowner=shapiramlrun/client_version=1.11.0-rc40mlrun/client_python_version=3.11.14host=test-custom-func-custom-gkvz2,,,,test_file


> 2026-03-08 13:50:22,696 [info] Run execution finished: {"name":"test-custom-func-custom","status":"completed"}


## 6. Unpack Custom packager
Expcted result -
* unpack the zip file to the temp folder and check the file content and run without assert issue

In [50]:
func = project.set_function("./test-custom-unpack.py",name="test-custom-unpack",kind='job',image='mlrun/mlrun',handler='func_custom',with_repo=True)

In [51]:
input_zip = test_func.outputs["test_file"]

In [52]:
func.run(inputs={"zip_file":input_zip},returns=["State"])

> 2026-03-08 13:50:22,758 [info] Storing function: {"db":"http://mlrun-api:8080","name":"test-custom-unpack-func-custom","uid":"f7c0c97f52224932b40f48372f5a0ee2"}


Unexpected run keyword argument 'local' was ignored.


> 2026-03-08 13:50:23,023 [info] Job is running in the background, pod: test-custom-unpack-func-custom-66bbd
> 2026-03-08 13:50:25,887 [info] extracting source from v3io:///projects/test-pack-shapira/artifacts/source-zip.zip to /mlrun/code

> 2026-03-08 13:50:26,111 [info] downloading v3io:///projects/test-pack-shapira/artifacts/test-custom-func-custom/0/test_file.zip to local temp file
/tmp/tmpchj3865s-2026-03-08T13:50:26.113583+00:00
> 2026-03-08 13:50:26,114 [warning] Skipping logging an object with the log hint 'key='State' tag='' itemized=False artifact_type=None packing_kwargs={} labels=None artifact_path=None extra_data={} metrics={}' due to the following error:
An exception was raised during the packing of 'State': The given path is not a file nor a directory: 'Completed'
> 2026-03-08 13:50:26,184 [info] To track results use the CLI: {"info_cmd":"mlrun get run f7c0c97f52224932b40f48372f5a0ee2 -p test-pack-shapira","logs_cmd":"mlrun logs f7c0c97f52224932b40f48372f5a0ee2 -p test-

project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results
test-pack-shapira,...2f5a0ee2,0,Mar 08 13:50:25,2026-03-08 13:50:26.172533+00:00,completed,run,test-custom-unpack-func-custom,v3io_user=shapirakind=jobowner=shapiramlrun/client_version=1.11.0-rc40mlrun/client_python_version=3.11.14host=test-custom-unpack-func-custom-66bbd,zip_file,,


> 2026-03-08 13:50:32,166 [info] Run execution finished: {"name":"test-custom-unpack-func-custom","status":"completed"}


In [53]:
db = mlrun.get_run_db()

In [54]:
db.delete_project(name=project.name,deletion_strategy='cascade')

> 2026-03-08 13:50:32,223 [info] Waiting for project to be deleted: {"project_name":"test-pack-shapira"}
> 2026-03-08 13:54:58,916 [info] Project deleted: {"project_name":"test-pack-shapira"}


In [55]:
shutil.rmtree('./context/')

In [56]:
shutil.rmtree('./custom_pack/')

In [57]:
os.remove('./test.zip')